# Context Managers & Resource Management (5+ Years Interview Guide)
Exhaustive revision guide to class-based context managers (__enter__/__exit__), @contextlib.contextmanager, contextlib.suppress, ExitStack, and async context managers on transaction data.

### Key 5-Year Interview Concepts Covered:
- **Class-Based Context Managers**: Dedicated cell for `__enter__()` and `__exit__()` (handling exceptions and suppression).
- **Generator Context Managers**: Dedicated cell for `@contextlib.contextmanager` and `try...finally`.
- **Utility Helpers**: Dedicated cell for `contextlib.suppress()` and `contextlib.ExitStack`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Class-Based Context Managers: `__enter__` & `__exit__`
**Explanation**: If `__exit__` returns `True`, exceptions raised inside the `with` block are swallowed (suppressed); if it returns `False` or `None`, exceptions propagate.

**Syntax**: `def __enter__(self): ... def __exit__(self, exc_type, exc_val, exc_tb): ...`

In [2]:
class TransactionLock:
    def __init__(self, tx_id):
        self.tx_id = tx_id
    def __enter__(self):
        print(f'[LOCK ACQUIRED] for {self.tx_id}')
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'[LOCK RELEASED] for {self.tx_id}')
        if exc_type is not None:
            print(f'[ERROR RECORDED] {exc_val}')
            return True # Suppress error
        return False

with TransactionLock('TX_999'):
    print('Processing atomic transaction...')
    raise ValueError('Simulated settlement error (suppressed by __exit__)')

[LOCK ACQUIRED] for TX_999
Processing atomic transaction...
[LOCK RELEASED] for TX_999
[ERROR RECORDED] Simulated settlement error (suppressed by __exit__)


### Generator-Based Context Managers: `@contextlib.contextmanager`
**Explanation**: Converts generator functions into context managers using `yield` and `try...finally` blocks.

**Syntax**: `@contextlib.contextmanager def timer(): ...`

In [3]:
@contextlib.contextmanager
def benchmark_block(task_name):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        t1 = time.perf_counter()
        print(f'{task_name} finished in {(t1-t0)*1000:.3f} ms')

with benchmark_block('Scanning 15k Transactions'):
    total = sum(float(t['transaction_amount']) for t in transactions)
    print(f'Total Spend Calculated: ${total:,.2f}')

Scanning 15k Transactions finished in 0.017 ms
Traceback (most recent call last):
  File "C:\Users\DELL\investigate-pandas\scratch\execute_and_populate_outputs.py", line 34, in execute_notebook
    exec(code, exec_globals)
  File "<string>", line 11, in <module>
  File "<string>", line 11, in <genexpr>
ValueError: could not convert string to float: ''


### Context Utilities: `contextlib.suppress` & `ExitStack`
**Explanation**: `suppress(*exceptions)` cleanly ignores specific errors. `ExitStack` programmatically manages dynamic sets of context managers.

**Syntax**: `with contextlib.suppress(KeyError): ...` / `with ExitStack() as stack: ...`

In [4]:
with contextlib.suppress(FileNotFoundError):
    os.remove('non_existent_file.tmp')
print('Suppressed FileNotFoundError cleanly.')

with contextlib.ExitStack() as stack:
    f1 = stack.enter_context(open(csv_path, 'r'))
    print('ExitStack opened CSV safely. First line len:', len(f1.readline()))

Suppressed FileNotFoundError cleanly.
ExitStack opened CSV safely. First line len: 151


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Re-entrant vs Non-Reentrant Context Managers
**Explanation**: Explain that context managers instantiated once can only be entered multiple times if their internal state resets properly in `__enter__`.

**Syntax**: `class ReentrantContext: ...`

In [5]:
print('Re-entrant context managers can be safely shared across nested or threaded `with` blocks.')

Re-entrant context managers can be safely shared across nested or threaded `with` blocks.
